In [1]:
inputs = [35,25]

In [2]:
type(inputs)

list

In [3]:
weights = [0.8, 0.1]

# **Sum Function**

In [4]:
def sum_func(inputs: list, weights: list):
    # res = 0
    # for _input, _weight in zip(inputs, weights):
    #     res += _input * _weight
    return sum(input_ * weight_ for input_, weight_ in zip(inputs, weights))

In [5]:
sum_func(inputs, weights)

30.5

# **Step Function**

In [6]:
def step_func(sum):
    return int(sum >= 1)

In [7]:
s = sum_func(inputs, weights)

In [8]:
step_func(s)

1

# **Using numpy to compute sum effectively**

In [9]:
import numpy as np

In [10]:
def sum_func(inputs, weights):
    return np.array(inputs) @ np.array(weights)

In [11]:
sum_func(inputs, weights)

np.float64(30.5)

## **Gradient Descent: First Attempt at Implementation**

In [ ]:
import numpy as np

def grad(features, rows, learning_rate):
    """
    assuming features = no. of features,
    rows is in the shape [[[x1, x2, x3, ...], y], [...], [...]],
    and learning rate is... learning rate :)
    """
    weights = np.random.randn(features)
    bias = 0

    for row in rows:
        x, y = row

        prediction = weights @ x + bias
        error = prediction - y

        djdw = (error) * (x) * 2.0
        djdb = (error) * 2.0

        weights -= djdw * learning_rate
        bias -= djdb * learning_rate
    return weights, bias

In [ ]:
# testing with f(x) = 3x + 5
def target_func(x):
    return 3 * x + 5

In [14]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [15]:
print(grad(1, data, 0.1))

(array([-1.06730528e+242]), np.float64(-1.078080532393578e+240))


## **Bad results because no epochs and high learning rate**

In [16]:
import numpy as np

def grad(features, rows, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    for epoch in range(epochs):

        total_loss = 0

        for x, y in rows:

            prediction = weights @ x + bias
            error = prediction - y

            total_loss += error**2

            djdw = 2 * error * x
            djdb = 2 * error

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

        if epoch % 100 == 0:
            print("Epoch:", epoch, "MSE:", total_loss / len(rows))

    return weights, bias

In [17]:
# testing with f(x) = 3x + 5

def target_func(x):
    return 3 * x + 5

In [18]:
data = [
    (np.array([x], dtype=float), target_func(x))
    for x in range(1, 100)
]

In [19]:
print(grad(1, data, 0.0001))

Epoch: 0 MSE: 279.89556861650254
Epoch: 100 MSE: 1.7758080486563634
Epoch: 200 MSE: 1.094424160819296
Epoch: 300 MSE: 0.6744897032600501
Epoch: 400 MSE: 0.4156855962164304
Epoch: 500 MSE: 0.25618554896632806
Epoch: 600 MSE: 0.15788623925521253
Epoch: 700 MSE: 0.0973047256050827
Epoch: 800 MSE: 0.059968555016214196
Epoch: 900 MSE: 0.036958406371017063
(array([3.0043172]), np.float64(4.5704492726519845))


## **Attempting to apply it to a real dataset**

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [21]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [22]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [23]:
x_train = transformations.fit_transform(x_train)

In [24]:
x_train

array([[ 0.4       , -0.6       , -0.33333333, ...,  0.        ,
         1.        ,  1.        ],
       [ 0.8       ,  0.        , -0.33333333, ...,  0.        ,
         0.        ,  2.        ],
       [-0.9       , -0.2       ,  0.66666667, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.1       ,  0.        ,  0.66666667, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        , ...,  0.        ,
         0.        ,  2.        ],
       [ 0.7       ,  0.5       , -0.33333333, ...,  0.        ,
         0.        ,  2.        ]], shape=(200000, 39))

In [25]:
x_test = transformations.transform(x_test)

In [26]:
train_data = list(zip(x_train, y_train))

In [27]:
test_data = zip(x_test, y_test)

In [28]:
run = False # this takes way too long, adviced not to run.
if run:
    weights, bias = grad(x_train.shape[1], train_data, learning_rate = 1e-6, epochs = 1000)

In [29]:
if run: 
    predictions = x_test @ weights + bias
    predictions.shape

In [30]:
if run:
    mse = sum(((y_test - predictions) ** 2)) / x_test.shape[0]
    mse

# **Optimizing for larger datasets**

In [31]:
import numpy as np

def grad(features, x, y, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    x, y = x.astype(np.float32), y.astype(np.float32)
    bias = 0.0

    for epoch in range(epochs):

        prediction = x @ weights + bias
        error = prediction - y

        djdw = (2 / len(x)) * (x.T @ error)
        djdb = (2 / len(x)) * np.sum(error)

        weights -= learning_rate * djdw
        bias -= learning_rate * djdb
        if epoch % 50 == 0:
            print(epoch, np.mean(error**2))
    return weights, bias

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("job_salary_prediction_dataset.csv")

In [33]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [34]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [35]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.to_numpy().reshape(-1,1)
).flatten()

In [36]:
x_train = transformations.fit_transform(x_train)

In [37]:
x_test = transformations.transform(x_test)

In [38]:
weights, bias = grad(x_train.shape[1], x_train, y_train_scaled, learning_rate = 0.008, epochs = 3000)

0 0.998343093637578
50 0.7524311124993343
100 0.6083847992017329
150 0.5093882232261779
200 0.43766584808092907
250 0.3832755942015428
300 0.340474288035619
350 0.3058175383035812
400 0.2771503441672982
450 0.25306152267771
500 0.23258314206513134
550 0.2150213348915385
600 0.19985869522692426
650 0.18669636033104886
700 0.17521851844376776
750 0.16516986708527376
800 0.15634073686520492
850 0.14855688191296348
900 0.14167220221257076
950 0.13556337229356838
1000 0.1301257548011465
1050 0.12527021171235966
1100 0.12092056442257221
1150 0.11701153758396433
1200 0.11348707338894957
1250 0.11029893596696515
1300 0.10740554717642023
1350 0.10477100967925038
1400 0.10236428336211659
1450 0.10015848847046056
1500 0.0981303142057199
1550 0.09625951560348076
1600 0.09452848464864953
1650 0.0929218840488347
1700 0.0914263340526017
1750 0.09003014428584104
1800 0.0887230838732912
1850 0.0874961841761506
1900 0.08634156935757595
1950 0.08525231072133266
2000 0.08422230138247591
2050 0.08324614834

In [39]:
predictions = x_test @ weights + bias   

In [40]:
predictions = y_scaler.inverse_transform(
    predictions.reshape(-1,1)
).flatten()

In [41]:
from sklearn.metrics import mean_absolute_error, r2_score

mean_absolute_error(y_test, predictions)

7369.190771752708

In [42]:
r2_score(y_test, predictions)

0.9287773072660881

# **Classification Model - Logistic Regression**

In [43]:
import numpy as np

# stochastic approach

def grad(features, rows, learning_rate, epochs=1000):

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    sigmoid = lambda x: 1 / (1 + np.e**(-x))

    for epoch in range(epochs):

        total_loss = 0

        for x, y in rows:

            prediction = weights @ x + bias # linear result
            prediction = sigmoid(prediction) # sigmoid squishes into probabilities
            prediction = np.clip(prediction, 1e-15, 1-1e-15) # prevent prediction = 1, sigmoid = -inf, and training error.
            error = prediction - y

            total_loss += -(y*np.log(prediction) + (1-y)*np.log(1-prediction))

            djdw = error * x
            djdb = error

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

        if epoch % 100 == 0:
            print("Epoch:", epoch, "Cross-Entropy Loss:", total_loss / len(rows))

    return weights, bias

In [44]:
def predict(x, weights, bias, threshold):
    sigmoid = lambda x: 1 / (1 + np.e**(-x))
    return 1 if sigmoid(weights @ x + bias) > threshold else 0

In [45]:
# data: [x1, x2]: if x1 + x2 >= 10, then 1. 0 <= x1, x2. <= 10

data = []

for _ in range(500):
    x1 = np.random.uniform(0, 10)
    x2 = np.random.uniform(0, 10)

    y = int(x1 + x2 >= 10)

    data.append((np.array([x1, x2]), y))

In [46]:
train_size = int(0.8 * 500)

train_data = data[:train_size]
test_data = data[train_size:]

In [47]:
weights, bias = grad(2, train_data, 0.1, 1000)

Epoch: 0 Cross-Entropy Loss: 1.0025059741011557
Epoch: 100 Cross-Entropy Loss: 0.0962658149349746
Epoch: 200 Cross-Entropy Loss: 0.08609425844356883
Epoch: 300 Cross-Entropy Loss: 0.08435640741114701
Epoch: 400 Cross-Entropy Loss: 0.08291057853438077
Epoch: 500 Cross-Entropy Loss: 0.08058502116008065
Epoch: 600 Cross-Entropy Loss: 0.07767880590325099
Epoch: 700 Cross-Entropy Loss: 0.0746062567407281
Epoch: 800 Cross-Entropy Loss: 0.0716246472846059
Epoch: 900 Cross-Entropy Loss: 0.06881178503160205


In [48]:
x_test, y_test = [row[0] for row in test_data], np.array([row[1] for row in test_data])

In [49]:
y_pred = np.array([predict(x, weights, bias, 0.5) for x in x_test])

In [50]:
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.99


In [51]:
weights, bias

(array([7.91841599, 7.67194906]), np.float64(-76.23307686844855))

In [52]:
import numpy as np  

# batch approach

def grad(x, y, learning_rate, epochs=1000):

    features = x.shape[1]

    weights = np.random.randn(features) * 0.01
    bias = 0.0

    x = x.astype(np.float32)
    
    sigmoid = lambda x: 1 / (1 + np.exp(-x))

    for epoch in range(epochs):

            prediction = x @ weights + bias # linear result
            prediction = sigmoid(prediction) # sigmoid squishes into probabilities
            prediction = np.clip(prediction, 1e-15, 1-1e-15) # prevent prediction = 1, sigmoid = -inf, and training error.

            error = prediction - y

            total_loss = -np.mean(y*np.log(prediction) + (1-y)*np.log(1-prediction))

            djdw = (x.T @ error) / x.shape[0]
            djdb = np.mean(error)

            weights -= learning_rate * djdw
            bias -= learning_rate * djdb

            if epoch % 100 == 0:
                        print("Epoch:", epoch, "Cross-Entropy Loss:", total_loss)

    return weights, bias

In [53]:
def predict(x, weights, bias, threshold = 0.5):
    sigmoid = lambda x: 1 / (1 + np.exp(-x))
    return np.array([sigmoid(x @ weights + bias) >= threshold])

In [54]:
# data: [x1, x2]: if x1 + x2 >= 10, then 1. 0 <= x1, x2. <= 10

x = []
y = []

for _ in range(500):
    x1 = np.random.uniform(0, 10)
    x2 = np.random.uniform(0, 10)

    label = int(x1 + x2 >= 10)

    x.append([x1, x2])
    y.append(label)

x = np.array(x)
y = np.array(y)

In [55]:
x.shape, y.shape

((500, 2), (500,))

In [56]:
train_size = int(0.8 * 500)

x_train, y_train, x_test, y_test = x[:train_size], y[:train_size], x[train_size:], y[train_size:]

In [57]:
for arr in [x_train, y_train, x_test, y_test]:
    print(arr.shape)

(400, 2)
(400,)
(100, 2)
(100,)


In [58]:
weights, bias = grad(x_train, y_train, 0.01, 5000)

Epoch: 0 Cross-Entropy Loss: 0.6781344022742468
Epoch: 100 Cross-Entropy Loss: 0.615096850476092
Epoch: 200 Cross-Entropy Loss: 0.5977715835039629
Epoch: 300 Cross-Entropy Loss: 0.5814898441139398
Epoch: 400 Cross-Entropy Loss: 0.5661831847152243
Epoch: 500 Cross-Entropy Loss: 0.5517865055944011
Epoch: 600 Cross-Entropy Loss: 0.538237675582209
Epoch: 700 Cross-Entropy Loss: 0.5254778025174711
Epoch: 800 Cross-Entropy Loss: 0.5134514035085863
Epoch: 900 Cross-Entropy Loss: 0.5021064735066539
Epoch: 1000 Cross-Entropy Loss: 0.49139447274821635
Epoch: 1100 Cross-Entropy Loss: 0.48127025229122394
Epoch: 1200 Cross-Entropy Loss: 0.47169193432976075
Epoch: 1300 Cross-Entropy Loss: 0.4626207612607605
Epoch: 1400 Cross-Entropy Loss: 0.454020924853147
Epoch: 1500 Cross-Entropy Loss: 0.44585938448226226
Epoch: 1600 Cross-Entropy Loss: 0.4381056813118934
Epoch: 1700 Cross-Entropy Loss: 0.4307317535537233
Epoch: 1800 Cross-Entropy Loss: 0.42371175649859155
Epoch: 1900 Cross-Entropy Loss: 0.4170218

In [59]:
y_pred = predict(x_test, weights, bias)

In [60]:
accuracy = np.mean(y_pred == y_test)

print("Accuracy:", accuracy)

Accuracy: 0.91


## **Attempting to implement a neural network**

In [ ]:
class neuron:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias
        
    def output(self, x):
        return self.sigmoid(x @ self.weights + self.bias)

    def sigmoid(self, x):   
        return 1 / (1 + np.exp(-x))
    

In [ ]:
class network:

    def __init__(self, layer_size, x, y, learning_rate = 0.01, epochs = 1000, threshold = 0.5, verbose = 0):
        """
        naming convention
    
            d()d() => partial diff
            ()f => final neuron related
            ()h => hidden neuron related
            j => cost func(cross entropy loss)
            p => sigmoid
            z => linear form(regression equation before segmoid)
            w => weights
            b => bias

        shapes
            x -> (samples, features)
            y -> (samples,)
            hidden_neurons -> (layer_size,)
            hidden_outputs -> (samples, layer_size)
            prediction -> (predictions,) or (samples,)
            error -> (errors,) or (samples,)
            weights -> (features,)
            final neuron weights -> (layer_size,)
            delta_hidden -> (samples, layer_size)
            
            """  
        features = x.shape[1]
        self.threshold = threshold
        self.hidden_neurons = list([neuron(np.random.randn(features) * 1.0, 0.0) for _ in range(layer_size)])
        self.final_neuron = neuron(np.random.randn(len(self.hidden_neurons)) * 1.0, 0.0)

        for epoch in range(epochs):
            hidden_outputs = self.hidden_outputs(x)
            prediction = self.final_neuron.output(hidden_outputs)
            error = prediction - y

            djdwf = (hidden_outputs.T @ error) / hidden_outputs.shape[0] # loss function wrt final neuron weights
            djdbf = np.mean(error) # loss function wrt final neurons bias

            delta_hidden = ((hidden_outputs * (1 - hidden_outputs)) * (self.final_neuron.weights * error[:, None])) 
            # loss function wrt hidden neuron outputs ^^

            djdwh = (x.T @ delta_hidden) / x.shape[0] # loss function wrt hidden neuron weights
            djdbh = np.mean(delta_hidden, axis = 0) # loss function wrt hidden neuron bias

            for n in range(len(self.hidden_neurons)):
                 self.hidden_neurons[n].weights -= djdwh[:, n] * learning_rate # since shape is (layer_size, 1)
                 self.hidden_neurons[n].bias -= djdbh[n] * learning_rate

            self.final_neuron.weights -= djdwf * learning_rate
            self.final_neuron.bias -= djdbf * learning_rate
            
            prediction = np.clip(
                prediction,
                1e-15,
                1-1e-15
            )

            loss = np.mean(
                -(y*np.log(prediction)
                +(1-y)*np.log(1-prediction))
            )

            if epoch % 1000 == 0:
                if verbose == 1:
                    print(epoch, loss)
                if verbose == 2:
                     print(epoch, loss)
                     print(np.max(np.abs(djdwf)))
                     print(np.max(np.abs(djdwh)))                 
            
    def hidden_outputs(self, x):
            outputs = np.array(list([n.output(x) for n in self.hidden_neurons]))
            return outputs.T

    def predict_proba(self, x):
        hidden_pred = self.hidden_outputs(x)
        prediction = self.final_neuron.output(hidden_pred)
        return prediction
    
    def predict(self, x):
        return (self.predict_proba(x) >= self.threshold).astype(np.int32)

In [63]:
x = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

y = np.array([
    0,
    1,
    1,
    0
], dtype=np.float32)

In [64]:
net = network(
    layer_size=4,
    x=x,
    y=y,
    learning_rate=0.1,
    epochs=10000
)

In [65]:
print(net.predict(x))

[0 1 1 0]


## **This is a single-layer neural network for binary classification**

In [10]:
# matrix style implementation; no neuron class

import numpy as np

class network:

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def __init__(self, layer_size, x, y, learning_rate = 0.01, epochs = 1000, intialization_strength = 0.01, threshold = 0.5, verbose = False):
        """
        naming convention
    
            d()d() => partial diff
            ()f => final neuron related
            ()h => hidden neuron related
            j => cost func(cross entropy loss)
            p => sigmoid
            z => linear form(regression equation before segmoid)
            w => weights
            b => bias

        shapes

            x => (samples, features)
            y => (samples,)
            hidden_weights => (features, layer_size)
            final_weights => (layer_size,)
            prediction => (samples,)
            hidden_outputs => (samples, layer_size)
            delta_hidden => (samples, layer_size)
            djdwh => (features, layer_size)
            """  
        
        self.threshold = threshold
        self.hidden_weights = np.random.randn(x.shape[1], layer_size) * intialization_strength
        self.hidden_biases = np.random.randn(layer_size) * intialization_strength

        self.final_weights = np.random.randn(layer_size) * intialization_strength
        self.final_bias = np.random.randn() * intialization_strength

        for epoch in range(epochs):
            hidden_outputs = self.sigmoid(x @ self.hidden_weights + self.hidden_biases)
            prediction = self.sigmoid(hidden_outputs @ self.final_weights + self.final_bias)
            prediction = np.clip(
                                prediction,
                                1e-15,
                                1-1e-15
                    )
            error = prediction - y
            m = x.shape[0]

            djdwf = hidden_outputs.T @ error / m
            djdbf = np.mean(error)

            delta_hidden = (hidden_outputs * (1 - hidden_outputs)) * (self.final_weights * error[:, None])

            djdwh = (x.T @ delta_hidden) / m
            djdbh = np.mean(delta_hidden, axis= 0)

            self.hidden_weights -= learning_rate * djdwh
            self.hidden_biases -= learning_rate * djdbh


            self.final_weights -= learning_rate * djdwf
            self.final_bias -= learning_rate * djdbf

            if verbose and epoch % 1000 == 0:
                loss = np.mean(
                                -(y*np.log(prediction)
                                +(1-y)*np.log(1-prediction))
                            )
                print(f"Epoch no. {epoch}; Cross-Entropy Loss: {loss}")

    def predict_proba(self, x):
        hidden_outputs = self.sigmoid(x @ self.hidden_weights + self.hidden_biases)
        prediction = self.sigmoid(hidden_outputs @ self.final_weights + self.final_bias)
        prediction = np.clip(
                            prediction,
                            1e-15,
                            1-1e-15
                            )
        return prediction

    def predict(self, x):
        probas = self.predict_proba(x)
        return (probas >= self.threshold).astype(np.int32)

In [11]:
x = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=np.float32)

y = np.array([
    0,
    1,
    1,
    0
], dtype=np.float32)

In [15]:
net = network(
    layer_size=4,
    x=x,
    y=y,
    learning_rate=1,
    intialization_strength=1,
    epochs=10000,
    verbose = 2
)

Epoch no. 0; Cross-Entropy Loss: 0.7839580404429576
Epoch no. 1000; Cross-Entropy Loss: 0.013796479185576981
Epoch no. 2000; Cross-Entropy Loss: 0.005284266540273301
Epoch no. 3000; Cross-Entropy Loss: 0.003193417071880214
Epoch no. 4000; Cross-Entropy Loss: 0.002266237628721826
Epoch no. 5000; Cross-Entropy Loss: 0.001747270771286769
Epoch no. 6000; Cross-Entropy Loss: 0.0014172236746140066
Epoch no. 7000; Cross-Entropy Loss: 0.0011895603469517166
Epoch no. 8000; Cross-Entropy Loss: 0.0010234097485521192
Epoch no. 9000; Cross-Entropy Loss: 0.0008970143709445795


In [21]:
print(list(format(y, ".25f") for y in net.predict_proba(x)))

['0.0007474876317431772146729', '0.9990243370148634838301405', '0.9994740171047749033306218', '0.0009405373637604096765871']
